### Prints all zotero attachments

In [1]:
from icecream import ic
import pathlib as pl
from collections.abc import Iterable
from pyzotero import zotero
from collections import defaultdict
import pandas as pd
import refwrangle as rfw
import matplotlib.pyplot as plt

In [2]:
def ensure_iterable(obj):
    """Wraps a scalar in a list if it's not iterable."""
    if isinstance(obj, str):  # Strings are technically iterable but should remain scalars
        return [obj]
    elif isinstance(obj, Iterable):
        return obj
    else:
        return [obj]
    
def get_first_creator(item):
    """Get first creator (usually author) from zotero db top level parent item"""
    # Check if creators key exists and is not empty
    if 'creators' in item['data'] and item['data']['creators']:
        creators = item['data']['creators']
        for creator in creators:
            if creator['creatorType'] == 'author':
                if 'name' in creator:
                    return creator['name']
                else:
                    return f"{creator['lastName']}, {creator['firstName']}"
    return ''  # no creators found

def get_item_venue(item):
    """Get the venue where item appeared.  Zotero puts this in many different fields"""
    data = item['data']
    
    # Check different possible venue fields in order of priority: I know these exist
    venueKeyInPriority = [
        'publicationTitle',  # For journal articles
        'journalAbbreviation', # For journal articles
        'bookTitle',         # For book chapters
        'publisher',         # For books
        'proceedingsTitle',  # For conference papers
        'blogTitle',         # For blog posts
        'websiteTitle',      # For web pages
        'encyclopediaTitle', # For encyclopedia articles
        'dictionaryTitle',   # For dictionary entries
        'conferenceName',    # For conference papers
        'university',        # For theses
        'publisher',         # For books
        'institution',       # For reports
        'libraryCatalog',    # For library catalog entries (how zotero files YouTube)
        'place',             # For location
    ]
    
    for field in venueKeyInPriority:
        if field in data and data[field]:
            return data[field] # assume field is the venue key
        
    # If here, didn't find any of the priority venues.
    # Try to find one in this possibly ficticious dict from perplexity
    # https://www.perplexity.ai/search/for-the-item-type-forum-post-w-_4Myygs7Qni_.0iVwQ3vLg#3

    venueKeyForItemType = {
        'blogPost': 'blogTitle',
        'book': 'publisher',
        'bookSection': 'bookTitle',
        'computerProgram': 'company',
        'conferencePaper': 'proceedingsTitle',
        'dataset': 'repository',
        'dictionaryEntry': 'dictionaryTitle',
        'document': 'archive',
        'email': 'subject',
        'encyclopediaArticle': 'encyclopediaTitle',
        'forumPost': 'forumTitle',
        'journalArticle': 'publicationTitle',
        'magazineArticle': 'publicationTitle',
        'manuscript': 'archive',
        'newspaperArticle': 'publicationTitle',
        'note': 'note',
        'preprint': 'repository',
        'presentation': 'conferenceName',
        'report': 'institution',
        'thesis': 'university',
        'videoRecording': 'libraryCatalog',
        'webpage': 'websiteTitle'
    }

    try:
        itemType = data['itemType']
        return venueKeyForItemType[itemType]
    except:
        print(f"failed to find venue for {rfw.get_citation_key(data)}")
        return ''

def get_parent_metadata(parent_item, collection_names):
    """Parse a parent_item's metadata into a dict w/ standardized names in the keys"""

    # Get "author": can be many things in zotero
    pdat = parent_item['data']
    firstCreator = ''
    if 'creators' in pdat and pdat['creators']:
        creators = pdat['creators']
        ctypes = [creator['creatorType'] for creator in creators]
        hasAuthor = 'author' in ctypes # so can prioritize author creator
        for creator in creators:
            if (creator['creatorType'] == 'author') or not hasAuthor:
                if 'name' in creator:
                    firstCreator = creator['name']
                else:
                    firstCreator = f"{creator['lastName']}, {creator['firstName']}"
                break

    def get_if_there(pkey):
        return pdat[pkey] if pkey in pdat else ''

    return dict(parentFirstCreator = firstCreator,
                # convert collections from zotero keys to names
                parentCollections=[collection_names[colkey] for colkey in pdat['collections']],
                
                parentCitekey=rfw.get_citation_key(pdat),
                parentVenue=get_item_venue(parent_item),
                parentDate = get_if_there('date'),
                parentTitle = get_if_there('title'),
                parentURL=get_if_there('url'),
                parentZotkey=pdat['key'])

In [3]:
# Read and parse the zotero database
zot = zotero.Zotero(rfw.library_id, rfw.library_type, rfw.api_key)
collection_names = (defaultdict(str) # handle missed collection (W5HNMQSX for parent QTESUD23)
                    | {collection['key']: collection['data']['name'] for collection in zot.collections()})

# Get all pdf and html attachments and associate them with their parent info
parentItems = zot.everything(zot.top())

In [7]:
child_exceptions, attachment_files = [], []
for parent in parentItems:
    if len(parent['meta']) < 1:
        continue # skip standalone notes or entries e.g. topictags.org

    pdat_always_save = get_parent_metadata(parent, collection_names)

    if rfw.is_youtube_video(parent):
        sourceInfo = pdat_always_save.copy()
        sourceInfo['contentType'] = 'youtube_video'
        attachment_files.append(sourceInfo) # not truly a file: info comes from URL
        continue
    
    parentCitekey = pdat_always_save['parentCitekey']
    errorParentIDstr = f'[{parentCitekey}]: {pdat_always_save['parentTitle']}'
    for child in zot.children(parent['key']):
        cdat = child['data'] | pdat_always_save
        if cdat['itemType'] != 'attachment':
            continue # I guess there's other stuff than attachments?

        fixed = {'Fixed':False}
        if cdat['title'] in ['PubMed entry', 'Semantic Scholar Link']:
            print(f'Skipping {cdat['title']}: {errorParentIDstr}')
            continue

        break
    break
    
#         try:
#             cdat['file_basename'] = cdat['path'].removeprefix("attachments:")
#             guessFNm = rfw.lit_attachment_dir_shared / cdat['file_basename']
#             if guessFNm.exists():
#                 cdat['file_fullpath'] = guessFNm
#             else:
#                 errStr = f'Error for {errorParentIDstr}. Full path does not exist: "{guessFNm}"'
#                 print(errStr)
#                 child_exceptions.append({'exception':errStr} | cdat | fixed)
#         except Exception as e:
#             print(f'Error for  {errorParentIDstr}: {e}')
#             # No idea why these errors occur.  Try to fix
#             for ext in ['pdf', 'html']: 
#                 guessBasename = f'{parentCitekey}.{ext}'
#                 guessFNm = rfw.lit_attachment_dir_shared / guessBasename
#                 if guessFNm.exists():
#                     cdat['file_basename'] = guessBasename
#                     cdat['file_fullpath'] = guessFNm
#                     break # if find pdf first, don't get html
#             if  'file_basename' in cdat:
#                 print(f"\tFix by guess basename worked: {cdat['file_basename']}")
#                 fixed = {'Fixed':True}
#             else:
#                 print(f'\tCould not fix it. Full path does not exist: "{guessFNm}"')
#                 fixed = {'Fixed':False}
#                 continue # no html or pdf: don't allow it in attachment_files (below)

#             child_exceptions.append({'exception': str(e)} | cdat | fixed)

#         attachment_files.append(cdat)

# attachment_files = pd.DataFrame(attachment_files)
# child_exceptions = pd.DataFrame(child_exceptions)

In [8]:
child

{'key': 'XNXUCHUM',
 'version': 21424,
 'library': {'type': 'user',
  'id': 60638,
  'name': 'scotto',
  'links': {'alternate': {'href': 'https://www.zotero.org/scotto',
    'type': 'text/html'}}},
 'links': {'self': {'href': 'https://api.zotero.org/users/60638/items/XNXUCHUM',
   'type': 'application/json'},
  'alternate': {'href': 'https://www.zotero.org/scotto/items/XNXUCHUM',
   'type': 'text/html'},
  'up': {'href': 'https://api.zotero.org/users/60638/items/XPGLFG5Y',
   'type': 'application/json'}},
 'meta': {},
 'data': {'key': 'XNXUCHUM',
  'version': 21424,
  'parentItem': 'XPGLFG5Y',
  'itemType': 'attachment',
  'linkMode': 'linked_file',
  'title': 'Mark257ChartsThat.html',
  'accessDate': '2025-01-22T07:01:08Z',
  'url': 'https://www.washingtonpost.com/business/2025/01/20/economy-sentiment-charts/',
  'note': '',
  'contentType': 'text/html',
  'charset': 'utf-8',
  'path': 'attachments:Mark25unhappyEcon7charts.html',
  'tags': [{'tag': 'zotmoov'}],
  'relations': {},
  'd

In [5]:
# unfixed = child_exceptions.query('Fixed != True')
# if (nUnfixed := len(unfixed)) > 0:
#     print(f'{nUnfixed} unfixed child exceptions')
#     display(unfixed)
# else:
#     print(f'Fixed {child_exceptions.Fixed.value_counts().values[0]} of {len(child_exceptions)} exceptions')

Fixed 130 of 130 exceptions


In [6]:
# # Find entries with unclassified venu

# unclassifiedVenues = []
# for parent in parentItems:
#     pinfo = get_parent_metadata(parent, collection_names)
#     if len(pinfo['parentVenue'])<1:
#         #print('Unclassified Venue:')
#         #display(pinfo)
#         unclassifiedVenues.append(pinfo)

# if (nUnclassifVenues := len(unclassifiedVenues)) > 0:
#     print(f'There were {nUnclassifVenues} unclassified venues:')
#     unclassifiedVenues = pd.DataFrame(unclassifiedVenues)
#     display(unclassifiedVenues)
# else:
#     print('No unclassified venues')

No unclassified venues


In [7]:
# # Count number of attachments (not parents) per collection
# citekeysInCollection = defaultdict(list)
# for row in attachment_files.itertuples(index=False):
#     for collection in row.parentCollections:
#         try:
#             citekeysInCollection[collection].append(row.parentCitekey)
#         except Exception as e:
#             ic(e, row)

# collection_counts = pd.Series({collection: len(filekeys) for collection, filekeys in citekeysInCollection.items()})
# collection_counts.sort_values(ascending=False, inplace=True)
# # plt.figure(figsize=(5, 10))  # Width is set to 8 inches, height to 5 inches
# # collection_counts.plot(kind='barh',)

In [8]:
# rfw.save_pickle_data(rfw.extractedZoteroEntriesFNm, 
#                  {'attachment_files':attachment_files, 
#                  'child_exceptions':child_exceptions,
#                  'collection_counts':collection_counts})

Writing to C:\Users\scott\OneDrive\share\ref\refwrangle\dat\zotero_entries.pkl...


In [9]:
# child_exceptions

,exception,key,version,parentItem,itemType,linkMode,title,accessDate,url,note,...,parentCitekey,parentVenue,parentDate,parentTitle,parentURL,parentZotkey,file_basename,file_fullpath,Fixed,path
0,'path',QCQ4TRTC,19176,59SLUUNU,attachment,imported_url,Basics of the Power Market | EPEX SPOT,2022-12-21T20:15:28Z,https://www.epexspot.com/en/basicspowermarket,,...,EPEX22basicsPowMkt,EPEX Spot,,Basics of the Power Market,https://www.epexspot.com/en/basicspowermarket,59SLUUNU,EPEX22basicsPowMkt.pdf,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,True,NaN
1,'path',MTV5GEQG,20259,S8NEX8A9,attachment,imported_url,Snapshot,2024-12-16T18:26:50Z,https://www.theatlantic.com/ideas/archive/2024...,,...,Friedersdorf24worstIdentPolit,The Atlantic,2024-12-15,How to Move On From the Worst of Identity Poli...,https://www.theatlantic.com/ideas/archive/2024...,S8NEX8A9,Friedersdorf24worstIdentPolit.html,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,True,NaN
2,'path',6ATW4QVV,19142,X33GKFGF,attachment,imported_file,Pratik24genAIuseCaseLegal.html,,,,...,Pratik24genAIuseCaseLegal,Intuz,28 Feb 2024,Generative AI in Legal: 5 Most Effective Use C...,https://www.intuz.com/blog/generative-ai-in-le...,X33GKFGF,Pratik24genAIuseCaseLegal.html,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,True,NaN
3,'path',3NRY7DW8,19632,F76I9F4J,attachment,imported_url,Snapshot,2024-12-09T21:45:59Z,https://pro.morningconsult.com/analysis/trump-...,,...,Yokley24immigLegalVSillegal,Morning Consult Pro,"December 09, 2024 at 5:00 am PST",Sentiment on Legal Migration Hasn’t Shifted Am...,https://pro.morningconsult.com/analysis/trump-...,F76I9F4J,Yokley24immigLegalVSillegal.html,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,True,NaN
4,name 'e' is not defined,Z4KI9NSQ,20309,SZ3FY7H4,attachment,linked_file,PDF,,,,...,Stephanopoulos24lessPolarized,Washington Post,2024-12-09,Surprise! America is less polarized than it us...,https://www.washingtonpost.com/opinions/2024/1...,SZ3FY7H4,Stephanopoulos24lessPolarized.html,C:\Users\scott\OneDrive\share\ref\obsidian\Obs...,True,attachments:Stephanopoulos24lessPolarized.pdf
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
125,name 'e' is not defined,JIPHE93A,14479,EKLZCYP5,attachment,linked_file,Bidgely19amInsightsRprt.pdf,,,,...,Bidgely19amInsightsRprt,"Bidgely, inc.",August 2019,AMI-Driven Insights Report,https://www.idcutilitiessummit.com/index/RESOU...,EKLZCYP5,C:\Users\scott\OneDrive\share\ref\zotero\paper...,NaN,True,C:\Users\scott\OneDrive\share\ref\zotero\paper...
126,name 'e' is not defined,62I2GBJK,14480,RCMMDNV6,attachment,linked_file,Mayhorn16disaggLdRealWrldPerf.pdf,,,,...,Mayhorn16disaggLdRealWrldPerf,ACEEE Summer Study on Energy Efficiency in Bui...,2016,Load Disaggregation Technologies: Real World a...,,RCMMDNV6,C:\Users\scott\OneDrive\share\ref\zotero\paper...,NaN,True,C:\Users\scott\OneDrive\share\ref\zotero\paper...
127,name 'e' is not defined,SX6AFLGY,14480,WA8IQAXP,attachment,linked_file,Hare18disaggHmLdDmdResp.pdf,,,,...,Hare18disaggHmLdDmdResp,Massachusetts Institute of Technology,2018,Disaggregation of residential home energy via ...,https://dspace.mit.edu/handle/1721.1/117983,WA8IQAXP,C:\Users\scott\OneDrive\share\ref\zotero\paper...,NaN,True,C:\Users\scott\OneDrive\share\ref\zotero\paper...
128,name 'e' is not defined,9G5G5RPX,14481,8MAAZN8P,attachment,linked_file,Rehman21LoadDisaggThesis.pdf,,,,...,Rehman21LoadDisaggThesis,Auckland University of Technology,2021,Load Disaggregation: Towards Energy Efficient ...,https://openrepository.aut.ac.nz/handle/10292/...,8MAAZN8P,C:\Users\scott\OneDrive\share\ref\zotero\paper...,NaN,True,C:\Users\scott\OneDrive\share\ref\zotero\paper...


In [10]:
# collection_counts[collection_counts> 6]

Generative AI                       314
Hot Takes US Elect 2024             273
                                    249
NeuroPsychoLinguisticPolitics       122
MediaAdsPolit                       119
Battery Review                       66
PoliticalML                          62
priceFrcstAEMO                       60
IdentityPolitics                     53
Conformal Prediction                 45
PowerMarkets                         44
battFires                            43
PollMethods                          38
Voting Systems                       30
Forecast aggr_disaggr                30
ElectionPredFeats                    28
Polarization                         26
CAISO Market                         25
copula                               23
MisDisinformation                    22
FocusGroups                          21
Contextual Optimization              18
priceSpikeVolatile                   17
Optimization                         17
ConceptDriftAdapt                    16
